[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/29_cnn/03_architectures_lenet_to_resnet.ipynb)

# Bài 03 — Kiến trúc CNN: từ LeNet đến ResNet

**🎯 Mục tiêu**
- Lịch sử các ý tưởng then chốt: LeNet, AlexNet, VGG, Inception, ResNet, MobileNet, EfficientNet.
- Các "viên gạch": chồng 3×3, **tích chập 1×1** (bottleneck), **depthwise separable**, **khối residual**.
- Thí nghiệm **degradation problem**: mạng "trơn" rất sâu khó train; kết nối tắt giải quyết.
- So sánh số tham số các mô hình trong `torchvision`; template model CNN.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/29_cnn

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import make_shapes
from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"                # bài này có mạng sâu 41 lớp — GPU giúp nhanh hơn nhiều
torch.backends.cudnn.deterministic = True                              # kết quả GPU tái lập được giữa các lần chạy
torch.backends.cudnn.benchmark = False
print("thiết bị:", device)


def n_params(m):
    return sum(p.numel() for p in m.parameters())

## 1. Dòng thời gian

| Năm | Kiến trúc | Ý tưởng chính | Top-5 lỗi ImageNet |
|---|---|---|---|
| 1998 | **LeNet-5** | conv → pool → conv → pool → FC; đọc chữ số trên séc ngân hàng | — |
| 2012 | **AlexNet** | sâu hơn, **ReLU**, dropout, augmentation, huấn luyện trên **GPU** | 15.3% |
| 2014 | **VGG** | chỉ dùng **3×3** xếp chồng, rất đều đặn (16–19 lớp) | 7.3% |
| 2014 | **GoogLeNet / Inception** | nhiều nhánh song song (1×1, 3×3, 5×5, pool); **1×1** giảm kênh | 6.7% |
| 2015 | **ResNet** | **kết nối tắt** (residual) → train được 152 lớp | 3.6% |
| 2017 | **MobileNet** | **depthwise separable** conv → mô hình nhẹ cho điện thoại | — |
| 2019 | **EfficientNet** | co giãn đồng thời độ sâu/rộng/độ phân giải | — |
| 2020+ | **ViT**, ConvNeXt | Transformer cho ảnh (chương 31); CNN "hiện đại hoá" | — |

## 2. LeNet-5 (phiên bản cho ảnh 28×28)

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2), nn.Tanh(), nn.AvgPool2d(2),    # 28 → 28 → 14
            nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2),               # 14 → 10 → 5
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
                                        nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


to_tensor = lambda a: torch.tensor(a[:, None], dtype=torch.float32)
X_tr, y_tr = make_shapes(3000, seed=0)
X_te, y_te = make_shapes(1000, seed=1)
Xtr, Xte, ytr, yte = to_tensor(X_tr), to_tensor(X_te), torch.tensor(y_tr), torch.tensor(y_te)


def train(model, epochs=8, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    history = []
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr), generator=g)
        for s in range(0, len(perm), 64):
            idx = perm[s:s + 64]
            opt.zero_grad()
            nn.functional.cross_entropy(model(Xtr[idx].to(device)), ytr[idx].to(device)).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            train_loss = nn.functional.cross_entropy(model(Xtr.to(device)), ytr.to(device)).item()
            test_acc = (model(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
        history.append((train_loss, test_acc))
    return history


torch.manual_seed(0)
lenet = LeNet5()
hist = train(lenet, epochs=10)
print(f"LeNet-5: {n_params(lenet):,} tham số | loss train = {hist[-1][0]:.3f} | accuracy test = {hist[-1][1]:.3f}")

## 3. Các viên gạch xây dựng

### Chồng 3×3 (VGG)
Hai lớp 3×3 có cùng trường tiếp nhận với một lớp 5×5 (bài 01) nhưng ít tham số hơn và thêm một phi tuyến:

In [ ]:
C = 64
print(f"một Conv 5×5 ({C}→{C}): {n_params(nn.Conv2d(C, C, 5)):,} tham số")
print(f"hai Conv 3×3 ({C}→{C}): {n_params(nn.Sequential(nn.Conv2d(C, C, 3), nn.Conv2d(C, C, 3))):,} tham số")

### Tích chập 1×1 và bottleneck
Conv 1×1 trộn thông tin **giữa các kênh** tại mỗi vị trí (như một lớp Linear áp dụng ở mọi pixel). Dùng để **giảm số kênh** trước phép 3×3 đắt đỏ — "bottleneck" của Inception và ResNet-50:

In [ ]:
direct = nn.Conv2d(256, 256, 3, padding=1)
bottleneck = nn.Sequential(nn.Conv2d(256, 64, 1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 256, 1))
x = torch.randn(1, 256, 14, 14)
print(f"Conv 3×3 trực tiếp 256→256: {n_params(direct):,} tham số | đầu ra {tuple(direct(x).shape)}")
print(f"bottleneck 1×1 → 3×3 → 1×1: {n_params(bottleneck):,} tham số | đầu ra {tuple(bottleneck(x).shape)}")

### Depthwise separable (MobileNet)
Tách conv chuẩn thành (1) **depthwise**: mỗi kênh một bộ lọc 3×3 riêng (`groups=C`), (2) **pointwise**: 1×1 trộn kênh. Chi phí giảm khoảng $\frac{1}{C_{out}} + \frac{1}{k^2}$ lần:

In [ ]:
C_in, C_out = 128, 128
standard = nn.Conv2d(C_in, C_out, 3, padding=1)
separable = nn.Sequential(nn.Conv2d(C_in, C_in, 3, padding=1, groups=C_in), nn.Conv2d(C_in, C_out, 1))
H = W = 28
flops = lambda m: sum(p.weight.numel() * H * W for p in m.modules() if isinstance(p, nn.Conv2d))   # phép nhân ≈ số trọng số × số vị trí
print(f"conv chuẩn:          {n_params(standard):,} tham số, ~{flops(standard) / 1e6:.1f} triệu phép nhân")
print(f"depthwise separable: {n_params(separable):,} tham số, ~{flops(separable) / 1e6:.1f} triệu phép nhân "
      f"(giảm {flops(standard) / flops(separable):.1f} lần; lý thuyết ≈ {1 / (1 / C_out + 1 / 9):.1f})")

### Khối Inception (minh hoạ)
Nhiều nhánh song song với kích thước kernel khác nhau, ghép theo chiều kênh — mạng tự chọn thang đo phù hợp:

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, c_in):
        super().__init__()
        self.b1 = nn.Conv2d(c_in, 16, 1)
        self.b3 = nn.Sequential(nn.Conv2d(c_in, 16, 1), nn.ReLU(), nn.Conv2d(16, 24, 3, padding=1))
        self.b5 = nn.Sequential(nn.Conv2d(c_in, 8, 1), nn.ReLU(), nn.Conv2d(8, 8, 5, padding=2))
        self.bp = nn.Sequential(nn.MaxPool2d(3, stride=1, padding=1), nn.Conv2d(c_in, 8, 1))

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


print("Inception: (1, 32, 14, 14) →", tuple(InceptionBlock(32)(torch.randn(1, 32, 14, 14)).shape), "(16 + 24 + 8 + 8 kênh)")

## 4. ResNet: vấn đề "suy thoái" và kết nối tắt

Thêm lớp lẽ ra không làm mạng **tệ hơn trên chính tập train** (các lớp thêm có thể học hàm đồng nhất). Thực tế, mạng "trơn" rất sâu lại có **loss train cao hơn** mạng nông — không phải overfitting mà là **khó tối ưu** (He et al., 2015).

**Khối residual**: học phần **dư** $F(x)$ thay vì cả ánh xạ: $y = \text{ReLU}(F(x) + x)$. Nếu lớp thừa, chỉ cần $F \approx 0$; gradient chảy thẳng qua nhánh tắt ($\partial y/\partial x = \partial F/\partial x + I$).

In [ ]:
class Block(nn.Module):
    def __init__(self, c, residual, bn=True):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=not bn)
        self.bn1 = nn.BatchNorm2d(c) if bn else nn.Identity()
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=not bn)
        self.bn2 = nn.BatchNorm2d(c) if bn else nn.Identity()
        self.residual = residual

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + x) if self.residual else torch.relu(out)


def deep_net(n_blocks, residual, bn=True, c=16):
    return nn.Sequential(nn.Conv2d(1, c, 3, padding=1), nn.BatchNorm2d(c) if bn else nn.Identity(), nn.ReLU(), nn.MaxPool2d(2),
                         *[Block(c, residual, bn) for _ in range(n_blocks)],
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 5))


rows = []
for bn, n_blocks in [(False, 2), (False, 10), (True, 20)]:
    for residual in [False, True]:
        torch.manual_seed(0)
        t0 = time.perf_counter()
        h = train(deep_net(n_blocks, residual, bn))
        rows.append((bn, 2 * n_blocks + 1, residual, h))
        print(f"{'có BN  ' if bn else 'không BN'} {2 * n_blocks + 1:2d} lớp conv, {'residual' if residual else 'trơn    '}: "
              f"loss train sau 8 epoch = {h[-1][0]:.3f} | accuracy test = {h[-1][1]:.3f} | {time.perf_counter() - t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for bn, depth, residual, h in rows:
    if depth > 5:
        ax.plot(range(1, 9), [a for a, _ in h], ls="-" if residual else "--",
                label=f"{depth} lớp, {'BN' if bn else 'không BN'}, {'residual' if residual else 'trơn'}")
ax.set_xlabel("epoch")
ax.set_ylabel("loss TRAIN")
ax.set_yscale("log")
ax.legend(fontsize=8)
plt.show()

- Không BN, mạng **trơn** 21 lớp không học được gì (loss đứng ở $\ln 5 \approx 1.61$ — đoán mò); thêm kết nối tắt thì học được.
- Có BN, mạng trơn 41 lớp học được nhưng **loss train** vẫn cao hơn rõ so với bản residual cùng độ sâu, và kết quả của mạng trơn rất thất thường (thử đổi seed hoặc thiết bị — có lần nó gần như không học) — đúng "degradation problem": vấn đề **tối ưu**, không phải overfitting.
- Residual + BN: 41 lớp train mượt. Đây là lý do mọi kiến trúc hiện đại (kể cả Transformer, chương 31) đều dùng kết nối tắt.

## 5. Model zoo trong `torchvision`

`torchvision.models` cài sẵn các kiến trúc (tạo với `weights=None` để không tải trọng số; chương 33 dùng trọng số pretrained cho transfer learning):

In [ ]:
try:
    import torchvision.models as tvm
    zoo = {"AlexNet": tvm.alexnet, "VGG-16": tvm.vgg16, "ResNet-18": tvm.resnet18, "ResNet-50": tvm.resnet50,
           "MobileNetV3-Small": tvm.mobilenet_v3_small, "EfficientNet-B0": tvm.efficientnet_b0}
    for name, fn in zoo.items():
        print(f"{name:18s}: {n_params(fn(weights=None)) / 1e6:6.1f} triệu tham số")
except ImportError:
    print("Cần torchvision để liệt kê model zoo.")

VGG-16 (2014) có ~138 triệu tham số, phần lớn ở các lớp fully-connected cuối; ResNet-50 chính xác hơn với ~25 triệu (bottleneck + global pooling); MobileNet/EfficientNet chính xác tương đương với vài triệu tham số — tiến bộ nằm ở **thiết kế**, không chỉ ở kích thước.

## 6. Template model: CNN

| Mục | Nội dung |
|---|---|
| Problem | phân loại/phát hiện/phân đoạn ảnh; tín hiệu dạng lưới (âm thanh, chuỗi 1D) |
| Intuition | bộ dò đặc trưng cục bộ chia sẻ trọng số, phân cấp cạnh → bộ phận → vật |
| Math | tích chập (cross-correlation), pooling, $H_{out} = \lfloor(H + 2p - k)/s\rfloor + 1$ |
| Algorithm | [Conv → BN → ReLU] × n, pooling/stride, kết nối tắt, global pooling, FC |
| Complexity | $\sim C_{in}C_{out}k^2HW$ phép nhân mỗi lớp; tham số $C_{out}(C_{in}k^2 + 1)$ |
| Hyperparameters | độ sâu/rộng, kernel, stride, augmentation, LR, weight decay |
| Assumptions | tính cục bộ và đồng biến dịch (thiên kiến quy nạp) |
| Strengths | hiệu quả dữ liệu với ảnh, ít tham số, transfer learning mạnh |
| Weaknesses | khó nắm quan hệ xa (trường tiếp nhận hữu hạn); nhạy với dịch chuyển phân phối |
| Failure cases | ảnh khác miền (nhiệt, y tế) mà không fine-tune; mạng trơn quá sâu |
| Production | transfer learning (chương 33), lượng tử hoá, MobileNet/EfficientNet cho thiết bị nhỏ |
| Alternatives | Vision Transformer, mô hình lai, MLP-Mixer |

## 🧠 Kiểm tra nhanh

1. AlexNet đóng góp những gì so với LeNet?
2. Vì sao VGG chỉ dùng 3×3? Conv 1×1 dùng để làm gì?
3. Depthwise separable conv tiết kiệm bao nhiêu và vì sao?
4. "Degradation problem" là gì? Vì sao không phải overfitting?
5. Khối residual giúp gradient lan truyền thế nào?

## 📌 Tóm tắt

- Tiến hoá CNN: sâu hơn (ReLU, GPU) → đều đặn (3×3) → nhiều nhánh/1×1 → residual → hiệu quả (depthwise, co giãn).
- Kết nối tắt giải quyết bài toán tối ưu mạng rất sâu; BN + residual là chuẩn mực.
- Số tham số không quyết định tất cả: thiết kế tốt cho mô hình nhỏ mà chính xác.

## 📚 Tài liệu tham khảo

- Krizhevsky, Sutskever & Hinton (2012) — *ImageNet Classification with Deep Convolutional Neural Networks*
- Simonyan & Zisserman (2015) — *Very Deep Convolutional Networks for Large-Scale Image Recognition*
- Szegedy et al. (2015) — *Going Deeper with Convolutions*
- He, Zhang, Ren & Sun (2016) — *Deep Residual Learning for Image Recognition*
- Howard et al. (2017) — *MobileNets*; Tan & Le (2019) — *EfficientNet*